# 03 — GAT-ViT Training
EfficientNetV2-S + dense 8-neighbor graph + GAT + Transformer on CIFAR-100.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
# !pip install torch torchvision timm torch-geometric torch-scatter torch-sparse matplotlib seaborn


In [ ]:
import torch
from src.models import GATViT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GATViT(num_classes=100).to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GAT-ViT trainable params: {n/1e6:.2f}M')

# Quick forward-pass sanity check
dummy = torch.randn(2, 3, 128, 128).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Output shape: {out.shape}  (expected: [2, 100])')

In [ ]:
# Visualize a patch graph for one image
import matplotlib.pyplot as plt
import networkx as nx
from src.graph_builder import PatchGraphBuilder
from src.models.gat_vit import MultiScaleEncoder

encoder = MultiScaleEncoder().to(device).eval()
builder = PatchGraphBuilder(patch_k=4, mode='dense')

with torch.no_grad():
    feat = encoder(dummy[:1])
    pyg = builder.build(feat)

# Extract first graph from batch
n_nodes = (pyg.batch == 0).sum().item()
edge_mask = (pyg.batch[pyg.edge_index[0]] == 0)
edges = pyg.edge_index[:, edge_mask].cpu().numpy()

G = nx.DiGraph()
G.add_nodes_from(range(n_nodes))
G.add_edges_from(zip(edges[0], edges[1]))

nH = feat.shape[2] // 4
nW = feat.shape[3] // 4
pos = {i: (i % nW, -(i // nW)) for i in range(n_nodes)}

fig, ax = plt.subplots(figsize=(8, 8))
nx.draw(G, pos=pos, ax=ax, node_size=50, node_color='steelblue',
        edge_color='gray', arrows=False, width=0.5)
ax.set_title(f'Dense Patch Graph ({n_nodes} nodes, {G.number_of_edges()} edges)')
plt.tight_layout()
plt.savefig('../results/dense_graph_viz.png', dpi=120)
plt.show()

In [ ]:
# !python ../src/train.py --model gat_vit --epochs 100 --batch 64 --out ../results --data ../data
from pathlib import Path
log_path = '../results/gat_vit/train_log.csv'
if Path(log_path).exists():
    from src.visualize import plot_training_curves
    plot_training_curves(log_path, Path('../results/gat_vit'))
    from IPython.display import Image
    display(Image('../results/gat_vit/training_curves.png'))
else:
    print('Training log not found. Run training first.')